# reranker_colab.ipynb — 리랭커(BAAI/bge-reranker-v2-m3)를 코랩 런타임에서 돌리기 위한 노트북

배경: 로컬 컴퓨터(RAM 7.4GB)에서는 bge-reranker-v2-m3(568M 파라미터)를 로드하려고 하면
세그멘테이션 폴트로 프로세스가 죽는다(2026-08-13 실측 확인) — 임베딩(e5-small/e5-large)은
디스크 공간을 확보한 뒤 정상 동작했지만, 리랭커는 모델 자체가 더 커서 로컬 RAM으로는
부족한 것으로 보인다.

이 노트북은 VSCode의 Colab 확장으로 이 파일에 연결해서, 무거운 모델(리랭커·임베딩)만
코랩의 더 넉넉한 런타임에서 실행하기 위한 것이다. `agent/` 패키지 코드는 건드리지 않는다 —
같은 코드를 어디서 실행하느냐만 다르게 하는 것이 목표(로컬 코드와 갈라지지 않게).

## 사용법
1. VSCode에서 이 노트북 열기 → 우측 상단 "커널 선택" → "Colab" → "New Colab Server" → 구글 계정 로그인
2. 아래 셀을 순서대로 실행해서 (1) 연결 확인 → (2) 리랭커 로드 확인까지 검증
3. 여기까지 되면, 다음 단계로 실제 배치 파이프라인과 연동하는 방법을 이어서 설계한다

## 1. 연결 확인 — 이게 코랩에서 도는지, 로컬에서 도는지부터 확인

In [1]:
import platform, os
print("플랫폼:", platform.platform())
print("코랩 여부(google.colab 모듈 있는지):", end=" ")
try:
    import google.colab  # noqa: F401
    print("코랩에서 실행 중")
except ImportError:
    print("코랩 아님 — 로컬에서 도는 중 (커널 선택이 안 된 상태일 수 있음)")

# 메모리 확인 (코랩이면 보통 12GB+, Pro면 더 많음)
!cat /proc/meminfo 2>/dev/null | head -3 || echo "(Linux 환경 아님 — Windows 로컬일 가능성)"

플랫폼: Linux-6.6.122+-x86_64-with-glibc2.35
코랩 여부(google.colab 모듈 있는지): 코랩에서 실행 중
MemTotal:       13286936 kB
MemFree:         8858664 kB
MemAvailable:   12252216 kB


## 2. 리랭커(bge-reranker-v2-m3) 로드 테스트
로컬에서 세그멘테이션 폴트로 죽었던 바로 그 모델. 여기서 정상 로드되는지 확인.

In [2]:
!pip install -q sentence-transformers

In [3]:
import time
from sentence_transformers import CrossEncoder

t0 = time.time()
model = CrossEncoder("BAAI/bge-reranker-v2-m3")
print(f"로드 성공! 소요: {time.time() - t0:.1f}초")

score = model.predict([("소매 판매가 늘었다", "소매판매액지수")])
print("테스트 점수:", score)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

로드 성공! 소요: 39.6초
테스트 점수: [0.6988919]


## 3. 실제 파이프라인 연동

로컬에서 아래 스크립트로 1~2단계(분류+claim추출)와 3단계 키워드매칭까지 끝내면
`data/rerank_pending.json`이 생긴다(claim별 `keyword_candidates`만 포함 — 임베딩/VDB
병합은 아직 안 함).

(2026-08-17: 2026-08-15에 임베딩 매칭을 로컬로 옮겼었는데, claim이 여러 건이면(2건만
있어도 재현됨) `batch_embedding_search()` 호출 중 세그폴트로 프로세스가 죽는 게 실측
확인돼서 다시 코랩으로 되돌렸다(아래 3-1 셀).

2026-08-18: VDB(KOSIS 표 28만7천여 개)를 로컬 Chroma에서 Supabase(pgvector)로 옮기면서
코랩에서도 인터넷으로 접근 가능해졌다 — 예전엔 "코랩에서 접근 불가라 건너뛴다"고 했던
VDB를 이제 여기서 같이 조회한다(아래 3-1 셀). 임베딩 모델도 e5-large -> e5-small로
바뀌었다(Supabase 무료 티어 용량 제한 대응, agent/mapping/embedding_search.py의
KOSIS_EMBEDDING_MODEL과 동일해야 함).)

이 파일을 아래 셀에서 업로드하면, 3-1에서 임베딩+VDB 매칭까지 마저 하고, 그다음 리랭커로
점수를 매겨 `rerank_results.json`을 다운로드해준다. 그걸 로컬 `data/` 폴더에 넣고
`python -m agent.pipeline.resume_after_rerank`를 실행하면 4~8단계가 마저 진행된다.

점수 계산 방식(시그모이드 → RRF 융합)은 `agent/mapping/reranker.py`의
`rerank()`/`_rrf_fuse()`랑 똑같이 맞춰뒀다(2026-08-18, "verified/unverified 이분법 +
top-5 승격" 방식을 RRF로 교체) — 로컬에서 리랭커가 직접 돌 때랑 결과가 갈라지지 않게.

In [4]:
import shutil
from google.colab import drive

drive.mount("/content/drive")

# 아래 SRC 경로를 본인 드라이브에 업로드한 rerank_pending.json 실제 경로로 바꾸세요.
# (구글 드라이브 웹사이트나 드라이브 앱에서 이 파일을 "내 드라이브" 최상위에 끌어다 놓으면
# 기본값 그대로 써도 됩니다.)
SRC = "/content/drive/MyDrive/rerank_pending.json"
pending_filename = "rerank_pending.json"
shutil.copy(SRC, pending_filename)
print("복사됨:", pending_filename)

Mounted at /content/drive
복사됨: rerank_pending.json


In [5]:
import json

with open(pending_filename, encoding="utf-8") as f:
    pending = json.load(f)

# 2026-08-16: 아래 리랭킹 셀의 doc_text_for()가 bare `catalog`를 참조하는데, 이 셀이
# pending["catalog"]만 있고 catalog라는 이름 자체를 assign한 적이 없어서
# NameError("catalog")가 났다(실측 확인). 여기서 명시적으로 꺼내둔다.
catalog = pending["catalog"]

print(f"리랭킹 대상 claim {len(pending['items'])}건, 카탈로그 {len(catalog)}개 표")

리랭킹 대상 claim 39건, 카탈로그 64개 표


## 3-1. VDB 매칭 (로컬에서 못 도는 부분만 여기서 실행)

2026-08-19: 골든셋 실측(Recall@5가 0%에 가까움)을 계기로 VDB(Supabase/pgvector, KOSIS 표
28만7천여 개) 임베딩 모델을 e5-small에서 **Qwen3-Embedding-4B(truncate_dim=1024)**로
바꿨다. 64개 카탈로그(keyword+embedding)는 그대로 e5-small이라 로컬(`export_for_rerank.py`)
에서 이미 병합까지 끝나서 `pending["items"][i]["merged_candidates"]`로 들어온다 — 이 셀은
**VDB만** 담당한다. Qwen3-Embedding-4B는 4B 파라미터라 로컬(RAM 7.4GB)에서 아예 못
돌아서(리랭커 568M도 이미 세그폴트였는데 그보다 7배 큼) 여기서만 처리할 수 있다.
Colab Pro+(GPU 우선 배정, 백그라운드 실행) 사용을 권장한다.

Supabase 연결 문자열은 노트북에 직접 쓰지 않는다(이 파일이 git에 커밋되므로 비밀번호
노출 위험). 원래 코랩 Secrets(열쇠 아이콘)를 쓰려고 했으나, 이 기능은 VSCode용 코랩
확장에서는 아직 지원 안 됨을 확인했다(웹 브라우저에서 직접 열었을 때만 가능) — 그래서
이미 쓰고 있는 "드라이브에 파일 올려두고 읽어오기" 패턴을 그대로 재사용한다.

**사전 준비(최초 1회만)**: 로컬에서 `{"db_url": ".env의 SUPABASE_DB_URL 값"}` 형식의
JSON을 `supabase_config.json`으로 만들어서 구글 드라이브 "내 드라이브" 최상위에
업로드해두세요(이 파일 자체는 git에 올리지 않음 — `.gitignore` 확인).

VDB에서 찾은 후보는 이미 있는 `merged_candidates`(keyword_rank/embedding_rank 태그 포함)에
`vdb_rank`만 추가로 얹는다. 합치는 규칙은 `agent/mapping/reranker.py`의
`_merge_candidates()`와 동일(2026-08-18, RRF 도입 이후): 각 소스에서의 순위(1부터)를
source_meta에 남겨서, 다음 셀(리랭킹)의 `rrf_fuse()`가 이 순위 + 리랭커 자체 순위를
합산해 최종 신뢰도를 계산하게 한다.

In [ ]:
!pip install -q sentence-transformers psycopg2-binary

import json as _json
import psycopg2
from sentence_transformers import SentenceTransformer

# 2026-08-19: 64개 카탈로그(keyword+embedding) 병합은 이제 export_for_rerank.py가 로컬에서
# 이미 끝내서 pending["items"][i]["merged_candidates"]에 들어있다 — 여기서 다시 만들
# 필요 없다(예전엔 이 셀이 e5-small로 64개 카탈로그를 다시 임베딩했는데, 로컬에서 이미
# 하는 걸 코랩에서 중복으로 하고 있었다). 이 셀은 이제 VDB(28만7천여 개, Qwen3-Embedding-4B,
# 4B라 로컬에서 못 돎)만 담당해서, 이미 있는 merged_candidates에 VDB 후보를 추가한다.
VDB_EMBED_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"
VDB_EMBED_DIM = 1024
vdb_embed_model = SentenceTransformer(VDB_EMBED_MODEL_NAME, truncate_dim=VDB_EMBED_DIM)
VDB_QUERY_INSTRUCTION = (
    "Given a Korean news claim sentence, retrieve the KOSIS statistical table "
    "description that best matches it"
)

VDB_TOP_K = 10
VDB_MIN_SIMILARITY = 0.5  # e5-small 기준으로 잡은 값 — Qwen으로 바뀌었으니 실측 후 재조정 필요할 수 있음
VDB_TABLE_NAME = "kosis_vdb_tables"

# 드라이브에 미리 올려둔 supabase_config.json({"db_url": "..."})을 읽는다 — 노트북
# 파일 자체(git 커밋됨)엔 연결 문자열을 절대 안 남긴다.
with open("/content/drive/MyDrive/supabase_config.json", encoding="utf-8") as f:
    _SUPABASE_DB_URL = _json.load(f)["db_url"]

_vdb_conn = None


def _get_vdb_connection():
    global _vdb_conn
    if _vdb_conn is None or _vdb_conn.closed:
        _vdb_conn = psycopg2.connect(_SUPABASE_DB_URL)
    return _vdb_conn


def vdb_query_vec_for(claim_sentence):
    # Qwen3-Embedding 공식 권장 사용법: 쿼리 쪽에만 instruction 프리픽스를 붙인다
    # (문서 쪽은 vdb_embedding_colab.ipynb에서 프리픽스 없이 그대로 인코딩했음).
    text = f"Instruct: {VDB_QUERY_INSTRUCTION}\nQuery: {claim_sentence}"
    return vdb_embed_model.encode([text], convert_to_numpy=True, normalize_embeddings=True)[0]


def vdb_candidates_for(query_vec):
    # agent/kosis/query_vdb.py의 batch_query_vdb()와 동일한 로직(코사인 거리 <=>,
    # 유사도=1-거리, 절대 유사도 하한선) — 여기 코랩에서도 같은 VDB(Supabase)를 조회한다.
    conn = _get_vdb_connection()
    with conn.cursor() as cur:
        cur.execute(
            f"""
            select tbl_id, text, embedding <=> %s::vector as distance
            from {VDB_TABLE_NAME}
            order by embedding <=> %s::vector
            limit %s;
            """,
            (query_vec.tolist(), query_vec.tolist(), VDB_TOP_K),
        )
        rows = cur.fetchall()

    candidates = []
    for tbl_id, text, dist in rows:
        similarity = 1.0 - float(dist)
        if similarity < VDB_MIN_SIMILARITY:
            continue
        candidates.append(
            {
                "table_id": tbl_id,
                "table_name": text,
                "score": similarity,
                "required_slots": [],
                "source_meta": f"kosis_vdb model={VDB_EMBED_MODEL_NAME}",
            }
        )
    return candidates


def _tag_source_rank(cand, key, rank):
    tag = f"{key}={rank}"
    meta = f"{cand['source_meta']} | {tag}" if cand.get("source_meta") else tag
    return {**cand, "source_meta": meta}


def add_vdb_to_merged(merged_candidates, vdb_cands):
    # export_for_rerank.py에서 이미 keyword_rank/embedding_rank가 태그된 채로 넘어온
    # merged_candidates에, vdb_rank만 추가로 얹는다(agent/mapping/reranker.py의
    # _merge_candidates()와 동일한 규칙 — RRF가 순위 태그만으로 신뢰도를 계산함).
    merged = {c["table_id"]: dict(c) for c in merged_candidates}
    for i, c in enumerate(vdb_cands):
        existing = merged.get(c["table_id"])
        if existing is None:
            merged[c["table_id"]] = _tag_source_rank(c, "vdb_rank", i + 1)
        else:
            existing["source_meta"] = f"{existing['source_meta']} | vdb_rank={i + 1}"
    return list(merged.values())


for i, item in enumerate(pending["items"]):
    vdb_query_vec = vdb_query_vec_for(item["claim"]["sentence"])
    vdb_cands = vdb_candidates_for(vdb_query_vec)
    item["merged_candidates"] = add_vdb_to_merged(item["merged_candidates"], vdb_cands)
    if i % 10 == 0:
        print(f"VDB 매칭 진행: {i}/{len(pending['items'])}")

print("VDB 매칭 + 병합 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 60.2 MB/s eta 0:00:0000:0100:01


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.3k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

VDB 매칭 진행: 0/39
VDB 매칭 진행: 10/39
VDB 매칭 진행: 20/39
VDB 매칭 진행: 30/39
VDB 매칭 + 병합 완료


In [7]:
import math
import re


def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))


# agent/mapping/reranker.py의 _rrf_fuse()/is_rrf_trusted()와 반드시 같은 로직으로 유지
# — 로컬 경로랑 코랩 경로의 판정 방식이 갈라지면 안 됨.
#
# 2026-08-18: 기존 "keyword=신뢰(verified) / embedding·vdb=unverified" 이분법 +
# 순위 기반 top-5 승격(promote_verified_within_top_ranks, VERIFIED_PROMOTION_RANK=5)을
# RRF(Reciprocal Rank Fusion)로 대체한다. 실측(울릉군 기사)에서 VDB 단독으로 찾은 표가
# keyword_search가 아예 못 찾은 진짜 정답이었던 사례("고용률(시/군/구)")가 확인됐는데,
# 기존 이분법은 소스 종류만 보고 이런 표를 무조건 "검증 안 됨"으로 버렸다. RRF는 각
# 소스(keyword/embedding/vdb)에서의 순위 + 리랭커 자체 순위를 모두 1/(k+rank)로 합산하는
# 방식이라, 특정 소스를 특별 취급하지 않고도 여러 소스에서 상위권으로 뽑힌 표가 자연스럽게
# 위로 올라온다.
RRF_K = 60  # RRF 원 논문(Cormack et al., 2009) 관례값.

_RANK_TAG_RE = re.compile(r"\b(keyword_rank|embedding_rank|vdb_rank|reranker_rank)=(\d+)")


def parse_rrf_ranks(source_meta):
    return {key: int(val) for key, val in _RANK_TAG_RE.findall(source_meta or "")}


def is_rrf_trusted(source_meta):
    # keyword_search가 찾았거나, 리랭커가 전체 후보 풀 중 독자적으로 1위로 평가했으면
    # (reranker_rank=1) 신뢰한다. 둘 다 아니면(embedding/VDB 단독 저순위) 신뢰하지 않는다.
    ranks = parse_rrf_ranks(source_meta)
    return "keyword_rank" in ranks or ranks.get("reranker_rank") == 1


def rrf_fuse(reranker_ranked, k=RRF_K):
    fused = []
    for i, c in enumerate(reranker_ranked):
        ranks = parse_rrf_ranks(c.get("source_meta"))
        ranks["reranker_rank"] = i + 1
        rrf_score = sum(1.0 / (k + r) for r in ranks.values())
        meta = f"{c.get('source_meta')} | reranker_rank={i + 1} rrf_score={rrf_score:.4f}"
        fused.append({**c, "score": rrf_score, "source_meta": meta})
    fused.sort(key=lambda c: c["score"], reverse=True)
    return fused


def doc_text_for(c):
    # 2026-08-15: merged_candidates엔 이제 64개 카탈로그 후보뿐 아니라 VDB(KOSIS 표
    # 28만7천여 개) 후보도 섞여 있는데, pending["catalog"]엔 64개 카탈로그 정보만 있어서
    # VDB 표 ID로 조회하면 KeyError가 났다(실측 확인). VDB 후보는 자기 자신의 table_name을
    # 이미 candidate dict에 들고 있으니 그걸 폴백으로 쓴다.
    entry = catalog.get(c["table_id"])
    if entry is not None:
        return entry["embedding_text"]
    return c.get("table_name") or c["table_id"]


output_items = []
for i, item in enumerate(pending["items"]):
    claim_sentence = item["claim"]["sentence"]
    candidates = item["merged_candidates"]
    if not candidates:
        output_items.append({"item_id": item["item_id"], "candidates": []})
        continue
    docs = [doc_text_for(c) for c in candidates]

    raw_scores = model.predict([(claim_sentence, d) for d in docs])

    reranked = []
    for c, raw in zip(candidates, raw_scores):
        reranked.append({**c, "score": sigmoid(float(raw)), "raw_rerank_score": float(raw)})
    reranked.sort(key=lambda c: c["score"], reverse=True)
    fused = rrf_fuse(reranked)

    output_items.append({"item_id": item["item_id"], "candidates": fused[:5]})
    if i % 10 == 0:
        print(f"리랭킹 진행: {i}/{len(pending['items'])}")

print("리랭킹 완료:", len(output_items), "건")

리랭킹 진행: 0/39
리랭킹 진행: 10/39
리랭킹 진행: 20/39
리랭킹 진행: 30/39
리랭킹 완료: 39 건


In [8]:
import shutil

with open("rerank_results.json", "w", encoding="utf-8") as f:
    json.dump({"items": output_items}, f, ensure_ascii=False, indent=2)

# files.download()는 이 VS Code<->코랩 연결에서는 성공 메시지만 찍히고 실제 브라우저
# 다운로드가 안 뜨는 문제가 있음(업로드 위젯 때와 동일한 제약, 2026-08-14 확인).
# 이미 마운트된 드라이브에 저장해서 드라이브 웹/앱에서 직접 받는 방식으로 우회.
DRIVE_OUT = "/content/drive/MyDrive/rerank_results.json"
shutil.copy("rerank_results.json", DRIVE_OUT)
print(f"저장 완료: {DRIVE_OUT}")
print("구글 드라이브(내 드라이브 최상위)에서 rerank_results.json을 다운로드해서")
print("로컬 data/ 폴더에 옮기고 resume_after_rerank.py를 실행하세요.")

저장 완료: /content/drive/MyDrive/rerank_results.json
구글 드라이브(내 드라이브 최상위)에서 rerank_results.json을 다운로드해서
로컬 data/ 폴더에 옮기고 resume_after_rerank.py를 실행하세요.


In [9]:
# 필터 없이 실제 유사도 분포 확인 (claim 3개만)
for item in pending["items"][:3]:
    qv = vdb_query_vec_for(item["claim"]["sentence"])
    conn = _get_vdb_connection()
    with conn.cursor() as cur:
        cur.execute(
            "select tbl_id, text, embedding <=> %s::vector as distance "
            "from kosis_vdb_tables order by embedding <=> %s::vector limit 10;",
            (qv.tolist(), qv.tolist()),
        )
        rows = cur.fetchall()
    print(item["claim"]["sentence"])
    for tbl_id, text, dist in rows:
        print(f"  sim={1-dist:.3f}  {text}  ({tbl_id})")
    print()


코로나 시기였던 2020년 인구주택총조사의 최종 응답률은 96.3%였다.
  sim=0.606  국가데이터처 (2019-11) 단체 참여 (지난 1년간, 주된 응답, 13세 이상 인구)  (DT_1SSSP051R)
  sim=0.603  과학기술정보통신부 (2015-05) 실태조사 전체 응답률(2007~2008)  (TX_10505_A001)
  sim=0.600  과학기술정보통신부 (2014-12) 기존응답자와 신규응답자의 응답률(2007)  (TX_10505_A002)
  sim=0.592  국가데이터처 (2019-11) 단체 참여 (지난 1년간, 복수응답, 13세 이상 인구)  (DT_1SSSP052R)
  sim=0.584  성평등가족부 (2020-10) 본 표본가구 여부별 응답률  (DT_MOGE_1001000414)
  sim=0.560  보건복지부 (2022-05) 응답자의 사회인구학적 분포(성과 연령에 대한 보정 이후)  (TX_117_2009_HB023)
  sim=0.560  국가데이터처 (2025-11) 조사망률(인구 천 명당)  (DT_20WBH022)
  sim=0.552  보건복지부 (2022-05) 응답자의 사회인구학적 분포(성과 연령에 대한 보정 이전)  (TX_117_2009_HB022)
  sim=0.551  성평등가족부 (2016-08) 시도별 응답률  (DT_MOGE_1001301030)
  sim=0.547  국가데이터처 (2024-11) 총조사가구 총괄(행정구역/거처의 종류/가구원수/사용방수별)  (DT_1GA0001)

‘하드 케이스’를 만나면 베테랑 현주씨가 출동한다.
  sim=0.461  대한민국통계연감 (2010-06) 접대부 검진 실적(접대부별)  (DT_999S_291057)
  sim=0.446  경기도 화성시 (2025-12) 한센사업대상자 현황  (DT_63601_K000006)
  sim=0.445  대한민국통계연감 (2010-06) 소년범 검사수사사건  (DT_999S_242052)
  sim=0.4